# Beyond the Sweat – Colab backend
Run the cells top to bottom. The last cell keeps running: it serves predictions to the website and saves every submission to GitHub.

In [ ]:
!pip -q install flask flask-cors
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
from google.colab import userdata
GITHUB_USER = "YOUR_GITHUB_USERNAME"
SITE_REPO   = "beyond-the-sweat"   # repo that hosts the website (GitHub Pages)
DATA_REPO   = "beyond-the-sweat"   # where submissions.csv is saved; use a separate PRIVATE repo for privacy
BRANCH      = "main"
TOKEN = userdata.get("GITHUB_TOKEN")   # Colab Secrets (key icon) -> fine-grained token, Contents: read & write

In [ ]:
import os, re, csv, json, time, base64, threading, subprocess, datetime, requests
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error

# ---------- model (same data as the original app) ----------
FEATURES = ["Heart Rate", "Age", "Workout Duration", "BMI"]
RAW = [[155,19,30,27.80,445],[145,21,25,20.80,314],[160,18,35,22.90,530],[150,22,40,28.70,582],[165,20,20,25.40,315],[164,25,35,18.41,532],[161,20,81,35.93,1195],[167,19,80,27.34,1111],[127,19,90,28.41,1048],[168,29,47,38.16,721],[125,18,50,30.26,525],[120,25,40,19.91,442],[141,29,38,27.52,496],[132,24,75,26.40,825],[167,26,61,30.89,937],[127,20,62,28.31,654],[157,24,62,25.83,816],[160,26,79,42.63,1162],[150,21,78,19.11,1072],[136,26,65,17.30,734],[128,19,68,23.93,730],[167,24,50,18.07,772],[131,25,85,23.17,1016],[151,21,76,32.69,1046],[125,19,49,22.78,506],[163,23,88,44.84,1318],[154,21,65,21.77,832],[135,27,105,32.93,1299],[141,25,53,20.88,627],[165,18,72,17.27,990]]
train = pd.DataFrame(RAW, columns=FEATURES + ["Calories"])
X, y = train[FEATURES], train["Calories"]
model = RandomForestRegressor(n_estimators=200, random_state=42).fit(X, y)
MAE = mean_absolute_error(y, cross_val_predict(RandomForestRegressor(n_estimators=200, random_state=42), X, y, cv=LeaveOneOut()))
print(f"Model ready. Typical error: {MAE:.0f} kcal")

# ---------- GitHub helpers ----------
H = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
def api(repo, path): return f"https://api.github.com/repos/{GITHUB_USER}/{repo}/contents/{path}"
def gh_get(repo, path):
    r = requests.get(api(repo, path), headers=H, params={"ref": BRANCH})
    if r.status_code == 404: return None, None
    r.raise_for_status(); j = r.json()
    return base64.b64decode(j["content"]).decode(), j["sha"]
def gh_put(repo, path, text, msg):
    _, sha = gh_get(repo, path)
    body = {"message": msg, "content": base64.b64encode(text.encode()).decode(), "branch": BRANCH}
    if sha: body["sha"] = sha
    requests.put(api(repo, path), headers=H, json=body).raise_for_status()

# ---------- submissions log (restored from GitHub so history is never overwritten) ----------
CSV_PATH, HEAD = "submissions.csv", ["time_utc", "name", "hr", "age", "dur", "bmi", "kcal"]
lock, dirty = threading.Lock(), threading.Event()
old, _ = gh_get(DATA_REPO, "data/submissions.csv")
with open(CSV_PATH, "w", newline="") as f:
    f.write(old if old else ",".join(HEAD) + "\n")
print("Existing submissions:", max(0, len(open(CSV_PATH).read().splitlines()) - 1))

def syncer():
    while True:
        dirty.wait(); time.sleep(20); dirty.clear()   # batch writes: one commit per 20 s at most
        try:
            with lock: text = open(CSV_PATH).read()
            gh_put(DATA_REPO, "data/submissions.csv", text, "Add new submissions")
        except Exception as e:
            print("GitHub sync failed:", e); dirty.set()

# ---------- web API ----------
def num(v, label, lo, hi, integer=True):
    try: x = float(str(v).replace(",", "."))
    except ValueError: raise ValueError(f"{label} must be a number.")
    if not lo <= x <= hi: raise ValueError(f"{label} must be between {lo} and {hi}.")
    return int(round(x)) if integer else x

app = Flask(__name__); CORS(app); hits = {}

@app.get("/health")
def health(): return jsonify(ok=True, mae=round(MAE, 1))

@app.post("/predict")
def predict():
    ip, now = request.headers.get("CF-Connecting-IP", request.remote_addr), time.time()
    hits[ip] = [t for t in hits.get(ip, []) if now - t < 3600]
    if len(hits[ip]) >= 60: return jsonify(error="Too many requests. Try again later."), 429
    hits[ip].append(now)
    d = request.get_json(silent=True) or {}
    try:
        hr, age = num(d.get("hr"), "Heart rate", 40, 220), num(d.get("age"), "Age", 10, 100)
        dur, bmi = num(d.get("dur"), "Duration", 1, 300), num(d.get("bmi"), "BMI", 10, 60, False)
    except ValueError as e:
        return jsonify(error=str(e)), 400
    kcal = float(model.predict(pd.DataFrame([[hr, age, dur, bmi]], columns=FEATURES))[0])
    name = re.sub(r"[^\w .\-]", "", str(d.get("name", ""))[:40]).strip()
    if name and name[0] in "=+-@": name = "'" + name        # block spreadsheet formula injection
    row = [datetime.datetime.utcnow().isoformat(timespec="seconds"), name, hr, age, dur, bmi, round(kcal, 1)]
    with lock, open(CSV_PATH, "a", newline="") as f: csv.writer(f).writerow(row)
    dirty.set()
    return jsonify(kcal=kcal, low=max(0, kcal - MAE), high=kcal + MAE)

In [ ]:
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()
threading.Thread(target=syncer, daemon=True).start()

# public HTTPS address for the Colab server (free Cloudflare quick tunnel)
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
                        stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)
URL = None
for line in proc.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: URL = m.group(0); break
threading.Thread(target=lambda: [_ for _ in proc.stderr], daemon=True).start()

# tell the website where the backend is this session
gh_put(SITE_REPO, "config.json",
       json.dumps({"url": URL, "updated": datetime.datetime.utcnow().isoformat(timespec="seconds")}),
       "Update backend URL")
print("Backend live at", URL)
print("The website finds it within a couple of minutes (GitHub Pages redeploys). Keep this cell running.")
while True: time.sleep(30)